In [ ]:
%matplotlib ipympl

from helper import *

## min curvature polyfit


Know $p(x) \in \mathcal{P}^4$, $p(0) = v_0$, $p'(0) = v_1$, $p''(0) = v_2$, and $p(T) = v_3$.
Want $\min \int_0^T |p''|^2$.

In [ ]:
T = sp.Symbol("T", positive=True)
x = sp.Symbol("x")
a = [sp.Symbol(f"a_{i}") for i in range(5)]
v = [sp.Symbol(f"v_{i}") for i in range(4)]
p = sum([a[i] * x**i for i in range(len(a))])
pv = p.subs({
    a[0]: v[0],
    a[1]: v[1],
    a[2]: v[2] / 2,
})
a3 = sp.solve(sp.Eq(pv.subs(x, T), 0), a[3])[0]
pv = pv.subs({a[3]: a3})
pv

In [ ]:
p_int = sp.integrate(pv**2, (x, 0, T))
a4 = sp.simplify(sp.solve(sp.Eq(p_int.diff(a[4]), 0), a[4])[0])
a4

In [ ]:
p_res = pv.subs({a[4]: a4})
p_res

In [ ]:
p_res_lam = sp.lambdify([T] + v + [x], p_res, cse=True, docstring_limit=None, modules=["jax"])
print(p_res_lam.__doc__)

In [ ]:
def p_eval_fun(T, v_0, v_1, v_2, v_3, x):
    x0 = (1/2)*v_2
    x1 = T**2
    x2 = (39/10)*T*v_1 + (81/10)*v_0 + (3/4)*v_2*x1
    x3 = T**(-3.0)
    return v_0 + v_1*x + x**3*(-v_0*x3 - v_1/x1 - x2*x3 - x0/T) + x**2*x0 + x**4*x2/T**4

In [ ]:
# visual check
p_eval = functools.partial(p_eval_fun, 2.0, 1.0, -1.0, -1.0, 0.0)
t = np.linspace(-0.5, 2.5, num=2**10)
p_vals = p_eval(t)
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(t, p_vals)
ax.grid()

## quantitative analysis

In [ ]:
lti_int, x_data, y_data, ctrl_data = get_data(0)

In [ ]:
def pred_fun(hist, t, _):
    assert hist.shape == (3,)
    v0 = hist[-1]
    v1 = (hist[-1] - hist[-2]) / spec.dt
    v2 = jnp.squeeze(jnp.diff(jnp.diff(hist))) / spec.dt**2
    return p_eval_fun(2.0, v0, v1, v2, 0.0, t)

pred_err(pred_fun, y_data, 3)

In [ ]:
def pred_fun_check(hist, t, idx):
    x0 = x_data[idx]
    _, y = lti_int(x0=x0, u=jnp.ones_like(t) * ctrl_data[idx])
    return y

pred_err(pred_fun_check, y_data, 2)

## visualize

In [ ]:
idx = 4290
t = np.linspace(0, 2.0, num=spec.n + 1, endpoint=True)

fig, axs = plt.subplots(2, 1, figsize=(7, 8))
axs[0].plot(y_data[idx: idx + spec.n + 1], label="y_data")
axs[0].plot(pred_fun(y_data[idx - 2: idx + 1], t, idx), label="pred")
axs[1].plot(y_data[idx: idx + spec.n + 1], label="y_data")
axs[1].plot(pred_fun_check(y_data[idx - 1: idx + 1], t, idx), label="flat_pred")
for ax in axs:
    ax.grid()
    ax.legend()